<a href="https://colab.research.google.com/github/shubhamsenani/Marketing-Analytics/blob/main/MA_Improving_Data_Quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MA Improving Data Quality

## Dummy Marketing Dataset

In [ ]:
import pandas as pd
import numpy as np

# Creating a dummy marketing dataset with intentional data quality issues
data = {
    "customer_id": [101, 102, 103, 104, 105, 105],  # Duplicate customer
    "age": [25, None, 40, 29, 150, 35],             # Missing + invalid age
    "gender": ["Male", "Female", "female", "M", "Unknown", None],
    "channel": ["Email", "Social Media", "Email", "Paid Search", "email", "Social Media"],
    "campaign_cost": ["500", "700", None, "400", "300", "700"],  # Stored as string
    "conversion": ["Yes", "No", "Yes", "yes", "No", "Yes"],
    "purchase_value": [2000, 0, 3000, 1500, None, 3000]
}

# Converting dictionary to DataFrame
df = pd.DataFrame(data)

# Display the dataset
df

,customer_id,age,gender,channel,campaign_cost,conversion,purchase_value
0,101,25.0,Male,Email,500,Yes,2000.0
1,102,NaN,Female,Social Media,700,No,0.0
2,103,40.0,female,Email,None,Yes,3000.0
3,104,29.0,M,Paid Search,400,yes,1500.0
4,105,150.0,Unknown,email,300,No,NaN
5,105,35.0,None,Social Media,700,Yes,3000.0


In [ ]:
# Display basic information about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     6 non-null      int64  
 1   age             5 non-null      float64
 2   gender          5 non-null      object 
 3   channel         6 non-null      object 
 4   campaign_cost   5 non-null      object 
 5   conversion      6 non-null      object 
 6   purchase_value  5 non-null      float64
dtypes: float64(2), int64(1), object(4)
memory usage: 468.0+ bytes


## Prominent Data Quality Issues in Marketing Data

### Issue 1: Missing Values (Null / NaN)

**What is the Issue?**

Missing values occur when data is not recorded for certain fields due to:

* Tracking failures
* Incomplete customer forms
* Integration issues between platforms

**Impact on Marketing Analytics**

* Incorrect ROI and Customer Acquisition Cost (CAC) calculations
* Biased averages and aggregate metrics
* Reduced usable sample size for customer segmentation
* Errors or instability in machine learning models

**Common strategies to address this issue include:**

* Filling numeric values using the mean or median
* Filling categorical values using the mode or a placeholder such as "Unknown"
* Dropping rows only when the proportion of missing data is insignificant and does not affect analysis quality


In [ ]:
# Check missing values before treatment
df.isnull().sum()

,0
customer_id,0
age,1
gender,1
channel,0
campaign_cost,1
conversion,0
purchase_value,1


In [ ]:
# Fill missing age values with the median age
df["age"] = df["age"].fillna(df["age"].median())

# Fill missing campaign cost values with the median campaign cost
df["campaign_cost"] = df["campaign_cost"].fillna(df["campaign_cost"].mode()[0])

# Fill missing purchase values with 0 (assumes no purchase)
df["purchase_value"] = df["purchase_value"].fillna(0)

# Fill missing gender values with 'Unknown'
df["gender"] = df["gender"].fillna("Unknown")

# Check missing values after treatment
df.isnull().sum()

,0
customer_id,0
age,0
gender,0
channel,0
campaign_cost,0
conversion,0
purchase_value,0


**Output Explanation (Markdown)**

* All missing values have been successfully handled
* Numeric fields now support aggregation and modeling
* Dataset is analytics-ready without losing rows

### Issue 2: Duplicate Records


**What is the Issue?**

Duplicate records occur when the same customer or transaction appears more than once in the dataset due to:

* Multi-channel customer interactions
* CRM and marketing platform integration issues
* Absence of a unique customer identifier

**Impact on Marketing Analytics**

* Inflated customer counts
* Overestimated reach and conversion rates
* Duplicate ad targeting and increased campaign costs

**Common strategies to address this include:**

* Identifying duplicates using unique identifiers (e.g., customer_id)
* Retaining only one record per customer
* Defining clear deduplication rules (first, last, or most complete record)

In [ ]:
# Check duplicates before removal
int(df.duplicated(subset="customer_id").sum())

1

In [ ]:
# Remove duplicate customer records
df = df.drop_duplicates(subset="customer_id", keep="first")

# Verify duplicates are removed
int(df.duplicated(subset="customer_id").sum())

0

**Output Explanation**

* Duplicate customer records have been removed
* Each customer now appears only once in the dataset
* Customer-level metrics such as reach and conversions are no longer inflated

### Issue 3: Inconsistent Categorical Values  (Male,male,M)

**What is the Issue?**

Inconsistent categorical values occur when the same category is recorded using different formats, spellings, or cases, such as:
* "Male" vs "M" vs "female"
* "Email" vs "email"

**Impact on Marketing Analytics**

* Fragmented reporting for the same category
* Incorrect channel and demographic analysis
* Misleading dashboards and summaries

**Common strategies to address this include:**

* Converting text to a standard case (lowercase)
* Mapping multiple representations to a single standard value
* Defining controlled vocabularies for categorical fields

In [ ]:
df.loc[:, "gender"] = df["gender"].str.lower()
df.loc[:, "channel"] = df["channel"].str.lower()

df.loc[:, "gender"] = df["gender"].replace({
    "m": "male",
    "female": "female",
    "unknown": "unknown"
})

In [ ]:
# Check cleaned categorical values
print(df["gender"].unique())
print(df["channel"].unique())

['male' 'female' 'unknown']
['email' 'social media' 'paid search']


**Output Explanation**

* Categorical values are now standardized
* Each category appears only once
* Channel and demographic analysis will now be accurate

### Issue 4: Invalid or Out-of-Range Values

**What is the Issue?**

Invalid values occur when data falls outside logical or acceptable ranges, such as:
* Age values greater than 100
* Negative costs or revenue values

**Impact on Marketing Analytics**

* Incorrect customer personas
* Distorted segmentation and targeting
* Reduced reliability of predictive models

**Common strategies to address this include:**

* Applying logical constraints to numeric fields
* Replacing invalid values with median or mean values
* Flagging invalid records for further review

In [ ]:
# Replace unrealistic age values with median age
df.loc[df["age"] > 100, "age"] = df["age"].median()

# Verify correction
df["age"].describe()

,age
count,5.000000
mean,32.800000
std,5.848077
min,25.000000
25%,29.000000
50%,35.000000
75%,35.000000
max,40.000000


**Output Explanation**

* All age values now fall within a realistic range
* Demographic analysis is no longer distorted by extreme values

### Issue 5: Incorrect Data Type


**What is the Issue?**

Incorrect data types occur when numeric or logical values are stored as text due to:
* CSV imports
* API data formatting issues
* Manual data entry errors

**Impact on Marketing Analytics**

* Calculations fail or produce incorrect results
* Aggregations and visualizations break
* Models cannot process the data correctly

**Common strategies to address this include:**

* Explicitly converting columns to correct data types
* Validating data types after import

In [ ]:
# Inspect current data types
df.dtypes

,0
customer_id,int64
age,float64
gender,object
channel,object
campaign_cost,object
conversion,object
purchase_value,float64


In [ ]:
# Convert campaign_cost to numeric
# errors='coerce' converts invalid entries to NaN
df.loc[:, "campaign_cost"] = pd.to_numeric(
    df["campaign_cost"], errors="coerce"
)

# Verify corrected data types
df.dtypes

,0
customer_id,int64
age,float64
gender,object
channel,object
campaign_cost,int64
conversion,object
purchase_value,float64


**Output Explanation**

* Numeric fields are now stored in numeric format
* Mathematical operations and aggregations can be performed safely

### Issue 6: Inconsistent Binary Encoding


**What is the Issue?**

Binary fields contain multiple representations of the same logical value, such as:
* "Yes", "yes", "No"

**Impact on Marketing Analytics**

* Incorrect conversion rate calculations
* Funnel analysis errors
* Inconsistent reporting

**Common strategies to address this include:**

* Mapping binary values to 0 and 1
* Enforcing strict binary standards

In [ ]:
df.loc[:,"conversion"] = df["conversion"].str.lower().map({
    "yes": 1, "no":0
})


In [ ]:
print(df["conversion"].unique())

[1.0 0.0 nan]


In [ ]:
# Convert conversion field to binary
df.loc[:, "conversion"] = (
    df["conversion"]
    .str.lower()
    .map({"yes": 1, "no": 0})
)

In [ ]:
# Check result
print(df["conversion"].unique())

[1 0]


**Output Explanation**

* Conversion data is now binary (0 and 1)
* Funnel and ROI calculations are now reliable

### Issue 7: Zero or Meaningless Values

**What is the Issue?**

Zero values may represent multiple meanings, such as:
* No purchase
* Refund
* Data entry error

**Impact on Marketing Analytics**

* Incorrect Average Order Value (AOV)
* Misleading revenue and profitability metrics

**Common strategies to address this include:**

* Explicitly defining what zero represents
* Flagging zero values for analysis or filtering

In [ ]:
# Identify zero purchase values
df[df["purchase_value"] == 0]

,customer_id,age,gender,channel,campaign_cost,conversion,purchase_value
1,102,35.0,female,social media,700,0,0.0
4,105,35.0,unknown,email,300,0,0.0


In [ ]:
# Remove rows where purchase_value is 0
df = df[df["purchase_value"] != 0]

# Verify removal
df

,customer_id,age,gender,channel,campaign_cost,conversion,purchase_value
0,101,25.0,male,email,500,1,2000.0
2,103,40.0,female,email,700,1,3000.0
3,104,29.0,male,paid search,400,1,1500.0


**Output Explanation**

* Zero-value transactions have been clearly identified
* Analysts can now decide whether to exclude or treat them separately

### Issue 8: Outliers

**What is the Issue?**

Outliers are extreme values that differ significantly from the majority of the data.

**Impact on Marketing Analytics**

* Skewed averages and KPIs
* Poor model performance
* Misleading strategic insights

**Common strategies to address this include:**

* Using statistical methods to detect outliers
* Capping, transforming, or removing extreme values

In [ ]:
# Calculate Interquartile Range (IQR)
Q1 = df["purchase_value"].quantile(0.25)
Q3 = df["purchase_value"].quantile(0.75)
IQR = Q3 - Q1

# Identify outliers using IQR method
outliers = df[
    (df["purchase_value"] < Q1 - 1.5 * IQR) |
    (df["purchase_value"] > Q3 + 1.5 * IQR)
]

# Display detected outliers
outliers

,customer_id,age,gender,channel,campaign_cost,conversion,purchase_value


**Output Explanation**

* Extreme values can be identified using summary statistics
* Further treatment can be applied based on business rules

## Final Clean Dataset

In [ ]:
df

,customer_id,age,gender,channel,campaign_cost,conversion,purchase_value
0,101,25.0,male,email,500,1,2000.0
1,102,35.0,female,social media,700,0,0.0
2,103,40.0,female,email,700,1,3000.0
3,104,29.0,male,paid search,400,1,1500.0
4,105,35.0,unknown,email,300,0,0.0
